<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/ai/02_machine_learning/reinforcement_learning/experiment_deep_q_learning_pytorch_simple_environment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np

# Simple Q Network
class QNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_size, 16),
            nn.ReLU(),
            nn.Linear(16, action_size)
        )

    def forward(self, x):
        return self.fc(x)

# Dummy environment
states = [0, 1]
actions = [0, 1]

def step(state, action):
    next_state = random.choice(states)
    reward = 1 if action == 1 else 0
    return next_state, reward

# Initialize
q_net = QNetwork(1, 2)
target_net = QNetwork(1, 2)
optimizer = optim.Adam(q_net.parameters(), lr=0.01)

memory = []

gamma = 0.9
epsilon = 0.2

# Training
for episode in range(100):
    state = random.choice(states)

    for _ in range(10):
        state_tensor = torch.tensor([[state]], dtype=torch.float32)

        if random.random() < epsilon:
            action = random.choice(actions)
        else:
            action = torch.argmax(q_net(state_tensor)).item()

        next_state, reward = step(state, action)

        memory.append((state, action, reward, next_state))

        if len(memory) > 10:
            batch = random.sample(memory, 5)

            for s, a, r, ns in batch:
                s_t = torch.tensor([[s]], dtype=torch.float32)
                ns_t = torch.tensor([[ns]], dtype=torch.float32)

                target = r + gamma * torch.max(target_net(ns_t)).item()
                pred = q_net(s_t)[0][a]

                loss = (pred - target) ** 2

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        state = next_state

    # update target network
    target_net.load_state_dict(q_net.state_dict())

print("Training complete")

Training complete
